# YP-RVC 训练页

在优云智算的 Jupyter 里从上往下运行。最后一格会一直转，训练页就开在 **7860**。

命令说明在同目录的 `优云智算部署说明.md`。请只用你有权使用的干声训练。

项目要先解压到 `/root/YP-RVC`。不要上传本机 Windows 的 `.venv`。


In [ ]:
import sys
from pathlib import Path

candidates = [Path.cwd(), Path("/root/YP-RVC"), Path("/root/rvc")]
root = None
for item in candidates:
    if (item / "webui.py").is_file() and (item / "requirements.txt").is_file():
        root = item.resolve()
        break
if root is None:
    raise SystemExit("找不到 webui.py。请先把项目解压到 /root/YP-RVC，再打开这个笔记本。")

Path("/tmp/yprvc_root.txt").write_text(str(root), encoding="utf-8")
Path("/tmp/yprvc_python.txt").write_text(sys.executable, encoding="utf-8")
print("项目目录:", root)
print("Python:", sys.executable)


## 检查 GPU 和权重

`cuda` 必须是 `True`。下面任一文件缺少，都说明上传包不完整。


In [ ]:
%%bash
set -euo pipefail
ROOT=$(cat /tmp/yprvc_root.txt)
PY=$(cat /tmp/yprvc_python.txt)
cd "$ROOT"
echo "项目目录: $ROOT"
"$PY" -c "import torch; print('torch', torch.__version__); print('cuda', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else '无可用 GPU')"
missing=0
for f in assets/hubert/hubert_base.pt assets/rmvpe/rmvpe.pt assets/pretrained_v2/f0G40k.pth assets/pretrained_v2/f0D40k.pth logs/mute/1_16k_wavs/mute.wav
do
  if [ ! -f "$f" ]; then
    echo "缺少 $f"
    missing=1
  else
    echo "存在 $f"
  fi
done
if [ "$missing" -ne 0 ]; then
  echo "权重不齐。请按说明重新打包上传，不要只拷贝代码。"
  exit 1
fi


## 第一次安装依赖

装过就可以跳过，这一格会自己检测。`fairseq` 要在实例上编译，十几分钟都正常。

当前 Python 没有 CUDA 版 PyTorch 时，这一格会停下来。换 Pytorch 基础镜像，不要在这里安装 torch。


In [ ]:
%%bash
set -euo pipefail
ROOT=$(cat /tmp/yprvc_root.txt)
PY=$(cat /tmp/yprvc_python.txt)
cd "$ROOT"

if ! "$PY" -c "import torch; assert torch.cuda.is_available()"; then
  echo "当前 Python 没有 CUDA 版 PyTorch。请换优云智算 Pytorch 基础镜像，不要在这里 pip 安装 torch。"
  exit 1
fi

if "$PY" -c "import fairseq,gradio,faiss,pyworld,parselmouth,librosa,soundfile" 2>/dev/null; then
  echo "依赖已经装好。直接运行最后一格启动。"
  exit 0
fi

apt-get update
apt-get install -y ffmpeg libsndfile1 build-essential

mkdir -p /tmp/yprvc-py
printf '%s\n' '#!/bin/sh' "exec \"$PY\" \"\$@\"" > /tmp/yprvc-py/python
printf '%s\n' '#!/bin/sh' "exec \"$PY\" -m pip \"\$@\"" > /tmp/yprvc-py/pip
chmod +x /tmp/yprvc-py/python /tmp/yprvc-py/pip
ln -sfn /tmp/yprvc-py/python /tmp/yprvc-py/python3
ln -sfn /tmp/yprvc-py/pip /tmp/yprvc-py/pip3
export PATH="/tmp/yprvc-py:$PATH"

bash tools/install_fairseq.sh
grep -v '^fairseq==' requirements.txt > /tmp/requirements.nofairseq.txt
"$PY" -m pip install -r /tmp/requirements.nofairseq.txt
echo "安装完成。接下来运行最后一格。"


## 启动训练页

运行前先在控制台放行 **7860**：

- 有公网 IP：实例「更多操作」→「配置防火墙」→ 添加 TCP `7860`
- 华北一 C： 「配置开放端口」里把 `7860` 加到 HTTP 端口

浏览器打开 `http://公网IP:7860`。华北一 C 用控制台给的域名，端口改成 `7860`。

这一格会一直运行。点停止才会关掉训练页。


In [ ]:
%%bash
set -euo pipefail
ROOT=$(cat /tmp/yprvc_root.txt)
PY=$(cat /tmp/yprvc_python.txt)
cd "$ROOT"
export PORT=7860
export PYTHONIOENCODING=utf-8
export RVC_DATASET_DIR="${RVC_DATASET_DIR:-$ROOT/datasets}"
export RVC_LOG_DIR="${RVC_LOG_DIR:-$ROOT/logs}"
mkdir -p "$RVC_DATASET_DIR" "$RVC_LOG_DIR" "$RVC_LOG_DIR/weights" "$ROOT/assets/weights"
echo "训练页监听 0.0.0.0:${PORT}"
echo "数据集: $RVC_DATASET_DIR"
echo "实验输出: $RVC_LOG_DIR"
echo "有公网 IP 时打开 http://公网IP:${PORT}"
echo "华北一C 用控制台域名，端口改成 ${PORT}"
echo "本格会一直运行。停止本格即关闭训练页。"
exec "$PY" "$ROOT/webui.py"
